<a href="https://colab.research.google.com/github/ndvphuc07/MONEY/blob/main/MONEY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import tensorflow as tf
import os
DATA_DIR = '/content/drive/MyDrive/dataset2'
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
EPOCHS = 20
print("Đang tải dữ liệu...")
train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_dataset.class_names
print(f"Đã tìm thấy các nhãn: {class_names}")

AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip('horizontal'),
  tf.keras.layers.RandomRotation(0.2),
  tf.keras.layers.RandomZoom(0.2),
])
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input
base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,),
                                               include_top=False,
                                               weights='imagenet')
base_model.trainable = False
inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])
MODEL_SAVE_PATH = "/content/drive/MyDrive/model_tien_vn.keras"

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(filepath=MODEL_SAVE_PATH, monitor='val_accuracy', save_best_only=True)
]
print("Bắt đầu huấn luyện mô hình...")
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    callbacks=callbacks
)

print(f"🎉 Hoàn tất! Model có độ chính xác cao nhất đã được lưu tại: {MODEL_SAVE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đang tải dữ liệu...
Found 2203 files belonging to 9 classes.
Using 1763 files for training.
Found 2203 files belonging to 9 classes.
Using 440 files for validation.
Đã tìm thấy các nhãn: ['100k', '10k', '1k', '200k', '20k', '2k', '500k', '50k', '5k']
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Bắt đầu huấn luyện mô hình...
Epoch 1/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 402s 2s/step - accuracy: 0.2604 - loss: 2.0630 - val_accuracy: 0.5500 - val_loss: 1.4071
Epoch 2/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 4s 80ms/step - accuracy: 0.5553 - loss: 1.2786 - val_accuracy: 0.7386 - val_loss: 0.9603
Epoch 3/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 4s 64ms/step - accuracy: 0.7022 - loss: 0.9296 - val_accuracy: 0.7386 - val_loss: 0.7992
Epoch 4/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 4s 71ms/step - accuracy: 0.7379 - loss: 0.8063 - val_accuracy: 0.7773 - val_loss: 0.7069
Epoch 5/20
56/56 ━━━━━━━━━━━━━━━━

In [ ]:
import gradio as gr
import tensorflow as tf
import numpy as np
from PIL import Image
MODEL_PATH = "/content/drive/MyDrive/model_tien_vn.keras"
CLASS_NAMES = ['100k', '10k', '1k', '200k', '20k', '2k', '500k', '50k', '5k']
CLASS_LABELS_PRETTY = {
    '1k': '1,000 VND', '2k': '2,000 VND', '5k': '5,000 VND',
    '10k': '10,000 VND', '20k': '20,000 VND', '50k': '50,000 VND',
    '100k': '100,000 VND', '200k': '200,000 VND', '500k': '500,000 VND'
}

try:
    print("Đang tải mô hình nhận diện, vui lòng đợi...")
    model = tf.keras.models.load_model(MODEL_PATH)
    print("✅ Tải mô hình thành công!")
except Exception as e:
    print(f"❌ LỖI: Không thể tải mô hình từ '{MODEL_PATH}'.")
    model = None
def predict_currency(image):
    if model is None:
        return {"Lỗi: Chưa tải được mô hình": 1.0}

    if image is None:
        return {"Vui lòng tải ảnh lên": 1.0}
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image)
    image = image.resize((224, 224))
    img_array = tf.keras.utils.img_to_array(image)
    img_array = tf.expand_dims(img_array, 0)
    predictions = model.predict(img_array)
    confidences = {}
    for i, class_name in enumerate(CLASS_NAMES):
        pretty_name = CLASS_LABELS_PRETTY.get(class_name, class_name)
        confidences[pretty_name] = float(predictions[0][i])
    return confidences
with gr.Blocks(theme=gr.themes.Soft(), title="Nhận Diện Tiền Việt Nam") as demo:
    gr.Markdown(
        """
        # 🇻🇳 Ứng dụng AI Phân Loại Tiền Việt Nam
        Chụp hoặc tải lên một bức ảnh tờ tiền. Hệ thống AI sẽ phân tích và đưa ra dự đoán mệnh giá kèm theo độ tự tin.
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(label="Ảnh đầu vào", type="numpy")
            submit_btn = gr.Button("🔍 Phân tích ảnh", variant="primary")

        with gr.Column(scale=1):
            output_labels = gr.Label(num_top_classes=3, label="Mệnh giá dự đoán")


    submit_btn.click(fn=predict_currency, inputs=input_image, outputs=output_labels)
    input_image.change(fn=predict_currency, inputs=input_image, outputs=output_labels)


if __name__ == "__main__":
    demo.launch(debug=True, share=True)

Đang tải mô hình nhận diện, vui lòng đợi...
✅ Tải mô hình thành công!


/tmp/ipykernel_2314/4158976670.py:37: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Nhận Diện Tiền Việt Nam") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a0ca5599f14c7c263d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
